In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dateutil.rrule import weekday
import seaborn as sns
from scipy.ndimage import label
from sklearn.metrics import classification_report

In [2]:
data=pd.read_csv('training_data_vt2025.csv')


<h1>Discriminant analysis: LDA, QDA </h1>
<h2> abstraction of the problem</h2>

<h2>introduction</h1>

<p>LDA is a generative classifier model assumes the data is normaly distributed. This requires our data to be normalized since many of the features, including(Temp, Dew, Humidity, Windspeed, Cloudcover, Visibility) have a wide range of values. In addition, due to lack of variability and meaningfullness, we droped the Snow and Snow Depth.</p> We have chosen to procceed with two methods
<ul>
<li> Z-score
<li> Min-Max
</ul>
<h1> Z-score </h1>
<p> The standard deviation shows how much the data deviates from the mean. a small standar deviation makes the data less spread out around the mean.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import metrics

In [4]:
df=pd.read_csv('training_data_vt2025.csv')
df

,hour_of_day,day_of_week,month,holiday,weekday,summertime,temp,dew,humidity,precip,snow,snowdepth,windspeed,cloudcover,visibility,increase_stock
0,5,5,1,0,0,0,-7.2,-15.0,53.68,0.000,0,0.0,16.3,31.6,16.0,low_bike_demand
1,21,4,1,0,1,0,-1.3,-12.8,40.97,0.000,0,0.0,23.9,85.7,16.0,low_bike_demand
2,21,3,8,0,1,1,26.9,21.8,73.39,0.000,0,0.0,0.0,81.1,16.0,low_bike_demand
3,1,6,1,0,0,0,3.1,-4.0,59.74,0.000,0,0.0,19.2,0.0,16.0,low_bike_demand
4,17,0,3,0,1,0,11.7,-11.4,18.71,0.000,0,0.0,10.5,44.6,16.0,low_bike_demand
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,3,5,6,0,0,1,21.5,19.4,87.68,0.000,0,0.0,10.6,24.4,16.0,low_bike_demand
1596,14,0,6,0,1,1,23.2,20.1,82.43,2.217,0,0.0,9.8,92.1,10.4,low_bike_demand
1597,13,0,3,0,1,1,13.9,-2.2,32.93,0.000,0,2.0,18.2,79.3,16.0,low_bike_demand
1598,14,5,3,0,0,1,11.7,-9.3,22.09,0.000,0,0.0,5.8,24.4,16.0,high_bike_demand


In [5]:
data.describe()

,hour_of_day,day_of_week,month,holiday,weekday,summertime,temp,dew,humidity,precip,snow,snowdepth,windspeed,cloudcover,visibility
count,1600.00000,1600.000000,1600.000000,1600.000000,1600.000000,1600.00000,1600.000000,1600.000000,1600.000000,1600.000000,1600.0,1600.000000,1600.000000,1600.000000,1600.000000
mean,11.37125,3.022500,6.468750,0.033125,0.710000,0.64375,15.210313,7.750750,63.927844,0.122042,0.0,0.042713,13.082500,64.322375,15.344125
std,6.94837,2.012965,3.454741,0.179019,0.453904,0.47904,9.264785,10.026459,19.079419,0.920600,0.0,0.421198,7.756652,32.748869,2.323737
min,0.00000,0.000000,1.000000,0.000000,0.000000,0.00000,-9.100000,-18.400000,15.850000,0.000000,0.0,0.000000,0.000000,0.000000,0.100000
25%,5.00000,1.000000,3.000000,0.000000,0.000000,0.00000,7.700000,-0.800000,47.845000,0.000000,0.0,0.000000,7.500000,28.800000,16.000000
50%,12.00000,3.000000,6.000000,0.000000,1.000000,1.00000,15.500000,8.300000,65.175000,0.000000,0.0,0.000000,12.300000,79.300000,16.000000
75%,17.00000,5.000000,9.000000,0.000000,1.000000,1.00000,23.200000,16.800000,79.955000,0.000000,0.0,0.000000,17.600000,92.800000,16.000000
max,23.00000,6.000000,12.000000,1.000000,1.000000,1.00000,35.600000,24.300000,99.890000,25.871000,0.0,6.710000,43.800000,100.000000,16.000000


In [6]:
dummy=pd.get_dummies(df,dtype=int)
df=pd.concat([df,dummy],axis=1)

df.head()

,hour_of_day,day_of_week,month,holiday,weekday,summertime,temp,dew,humidity,precip,...,dew,humidity,precip,snow,snowdepth,windspeed,cloudcover,visibility,increase_stock_high_bike_demand,increase_stock_low_bike_demand
0,5,5,1,0,0,0,-7.2,-15.0,53.68,0.0,...,-15.0,53.68,0.0,0,0.0,16.3,31.6,16.0,0,1
1,21,4,1,0,1,0,-1.3,-12.8,40.97,0.0,...,-12.8,40.97,0.0,0,0.0,23.9,85.7,16.0,0,1
2,21,3,8,0,1,1,26.9,21.8,73.39,0.0,...,21.8,73.39,0.0,0,0.0,0.0,81.1,16.0,0,1
3,1,6,1,0,0,0,3.1,-4.0,59.74,0.0,...,-4.0,59.74,0.0,0,0.0,19.2,0.0,16.0,0,1
4,17,0,3,0,1,0,11.7,-11.4,18.71,0.0,...,-11.4,18.71,0.0,0,0.0,10.5,44.6,16.0,0,1


<h1>Independent and Dependent Variables</h1>

The output shows that we have our original variables and the dummy variables. However, we do not need all of this
information. Therefore, we will create a dataset that has the X variables we will use and a separate dataset that will have our y
values. Below is the code.

In [7]:
features = ['hour_of_day','day_of_week','month','weekday','temp', 'dew', 'humidity', 'windspeed', 'cloudcover', 'visibility']
y=data['increase_stock']
X=data[features]

<h1>Train and Test Sets</h1>

The X dataset has our ten features and the y dataset has our dependent variable which is increase stock or decrease stock. We
can not split our data into a train and test set. The code is below.

The training data set consist of 70% and the test data set is consist of 30% of our actual data. 

In [8]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=0)

In [9]:
scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)

In [10]:
X_train=pd.DataFrame(X_train)
X_test=pd.DataFrame(X_test)

In [11]:
#sns.pairplot(X_train)

In [12]:
X_train.describe().round(2)

,0,1,2,3,4,5,6,7,8,9
count,1120.00,1120.00,1120.00,1120.00,1120.00,1120.00,1120.00,1120.00,1120.00,1120.00
mean,-0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.64,-1.51,-1.60,-1.58,-2.61,-2.60,-2.51,-1.67,-1.93,-6.30
25%,-0.92,-1.01,-1.02,-1.58,-0.79,-0.85,-0.83,-0.71,-1.12,0.29
50%,0.08,-0.01,-0.16,0.63,-0.00,0.06,0.06,-0.12,0.48,0.29
75%,0.80,0.99,1.00,0.63,0.84,0.91,0.83,0.58,0.89,0.29
max,1.66,1.49,1.57,0.63,2.20,1.65,1.89,3.90,1.10,0.29


<h1>Training On given data</h1>

In [13]:
clf=LDA()
clf.fit(X_train,y_train)
clf.score(X_test,y_test)

0.8333333333333334

In [14]:
y_pred=clf.predict(X_test)
y_pred

array(['low_bike_demand', 'low_bike_demand', 'high_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'high_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'high_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_demand', 'low_bike_demand',
       'low_bike_demand', 'low_bike_d

In [15]:
print(classification_report(y_test,y_pred))

                  precision    recall  f1-score   support

high_bike_demand       0.54      0.36      0.43        84
 low_bike_demand       0.87      0.93      0.90       396

        accuracy                           0.83       480
       macro avg       0.70      0.65      0.67       480
    weighted avg       0.81      0.83      0.82       480



In [16]:
fpr, tpr, thresholds = metrics.roc_curve(y_test,y_pred)
roc_auc = metrics.auc(fpr, tpr)
roc_auc
plt.clf()
plt.plot(fpr,tpr,label="ROC Curver (are=%0.2f)" % roc_auc)
plt.plot([0, 1], [0, 1], linestyle='k--')


ValueError: y_true takes value in {'high_bike_demand', 'low_bike_demand'} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.